# GRAPH MIXER: VRG

#### 1. find rxn site
#### 2. get partial structures (aryls)
#### 3. mixing

In [7]:
nrows = 50000
similarity_value = 0.8 # for clustering
n_iter=15 # for graph mixer

In [8]:
#file load
import pandas as pd

dataset_train = pd.read_csv('original_datasets/raw_train.csv', nrows=nrows)
dataset_val = pd.read_csv('original_datasets/raw_val.csv', nrows=int(nrows*0.2))

def split_reaction(smiles_reaction):
    inputs, output = smiles_reaction.split('>>')  # split with '>>'        
    return pd.Series([inputs, output])

dataset_train[['inputs', 'output']] = dataset_train.iloc[:, 2].apply(split_reaction)
dataset_val[['inputs', 'output']] = dataset_val.iloc[:, 2].apply(split_reaction)
smiles_columns = ['inputs', 'output']

print(dataset_val) #check

                   id class  \
0        US08329716B2   UNK   
1          US06051718   UNK   
2        US07504410B2   UNK   
3          US04960769   UNK   
4     US20110092505A1   UNK   
...               ...   ...   
4996  US20140194411A1   UNK   
4997  US20090149445A1   UNK   
4998     US08710243B2   UNK   
4999  US20130303532A1   UNK   
5000     US06518265B1   UNK   

                          reactants>reagents>production  \
0     O=C(O[C:1](=[O:2])[C:3]([F:4])([F:5])[F:6])C(F...   
1     CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:...   
2     O=C(O[C:1](=[O:2])[C:3]([F:4])([F:5])[F:6])C(F...   
3     CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:...   
4     CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:...   
...                                                 ...   
4996  O[C:1]1([CH:2]2[CH2:3][CH2:4]2)[c:5]2[c:6]([cH...   
4997  Br[c:1]1[cH:2][cH:3][cH:4][c:5]([CH:6]=[C:7]2[...   
4998  O=[CH:1][n:2]1[c:3](-[c:4]2[c:5]([CH3:6])[n:7]...   
4999  Br[c:1]1[n:2][c:3]([NH:4][CH2:5

In [9]:
#reaction_site
import augment_models.reaction_site as reaction_site

dataset_train[['class', 'fg_site']] = pd.DataFrame(
    dataset_train.apply(
        lambda row: reaction_site.get_reaction_center(row['inputs'], row['output'], depth=1)[:2], axis=1).tolist())
dataset_val[['class', 'fg_site']] = pd.DataFrame(
    dataset_val.apply(
        lambda row: reaction_site.get_reaction_center(row['inputs'], row['output'], depth=1)[:2], axis=1).tolist())

In [10]:
#clustering_fg
import re
from collections import defaultdict
from rdkit import Chem
from rdkit import rdBase
import augment_models.functionalizer_synt as functionalizer_synt

rdBase.DisableLog('rdApp.*')

def remove_atom_map(smiles):
    return re.sub(r":\d+", "", smiles)

def get_mol(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except:
        return None
        
def get_cluster_key(smiles):
    mol = get_mol(smiles)
    if mol:
        return Chem.MolToSmiles(mol, canonical=True)
    return None

reaction_groups = defaultdict(list)
reaction_groups_val = defaultdict(list)

for idx, row in dataset_train.iterrows():
    fg_raw = row['fg_site']
    if not fg_raw:
        continue

    fg_clean = remove_atom_map(fg_raw)
    key = get_cluster_key(fg_clean)
    assigned_key = None

    if key:
        is_assigned = False
        for key_old, group_rows in reaction_groups.items():
            existing_fg_clean = remove_atom_map(group_rows[0]['fg_site'])
            sim = functionalizer_synt.calculate_similarity(existing_fg_clean, fg_clean, 2)
            if sim >= similarity_value:
                reaction_groups[key_old].append(row)
                is_assigned = True
                break

        if not is_assigned:
            reaction_groups[key].append(row)

print(f"clusters (train): {len(reaction_groups)}")

for idx, row in dataset_val.iterrows():
    fg_raw = row['fg_site']
    if not fg_raw:
        continue

    fg_clean = remove_atom_map(fg_raw)
    key = get_cluster_key(fg_clean)

    if key:
        is_assigned = False
        for key_old, group_rows in reaction_groups_val.items():
            existing_fg_clean = remove_atom_map(group_rows[0]['fg_site'])
            sim = functionalizer_synt.calculate_similarity(existing_fg_clean, fg_clean, 2)
            if sim >= similarity_value:
                reaction_groups_val[key_old].append(row)
                is_assigned = True
                break

        if not is_assigned:
            reaction_groups_val[key].append(row)

print(f"clusters (val): {len(reaction_groups_val)}")

## this can be used further cluster-based changer system

clusters (train): 1365
clusters (val): 380


In [11]:
#functionalizer
from rdkit import Chem
from tqdm import tqdm
import copy
from rdkit.Chem import rdmolops
from rdkit.Chem import RWMol
import augment_models.distance as distance

fg_smarts_list = {
    'phenyl': 'c1ccccc1',         
    'naphthyl': 'c1cccc2c1cccc2',   
    'pyridine': 'c1ncccc1',           
    'thiophene': 'c1sccc1',     
    'furan': 'c1occc1',             
    'pyrrole': 'c1[nH]ccc1',              
    'pyrrole_C': 'c1[nC]ccc1',             
    'imidazole': 'c1nc[nH]c1',     
    'imidazole_C': 'c1cnc[nX1]1',       
    'thiazole': 'c1scnc1',  
    'oxazole': 'c1ocnc1',      
    'pyrazole': 'c1cn[nH]c1',              
    'pyrazole_C': 'c1cn[nX1]c1',          
    'pyrimidine': 'c1ncncn1',       
    'pyrazine': 'c1ncccn1',             
    'pyridazine': 'c1nnccc1',         
    'benzothiophene': 'c1cc2ccccc2s1',   
    'indole': 'c1cc2ccccc2[nH]1',    
    'indole_C': 'c1cc2ccccc2n1C',  
    'benzofuran': 'c1cc2ccccc2o1',      
    'quinoline':        'c1ccc(cccc2)c2n1',                 
    'isoquinoline':     'c(ncc1)c2c1cccc2',                 
    'quinoxaline':      'c(n1)cnc2c1cccc2',        
    'quinazoline':      'c(n1)ncc2c1cccc2',               
    'benzimidazole':    'c1nc2ccccc2[nH]1',           
    'benzimidazole_C':    'c(cc1)cc2c1n(C)cn2',           
    'benzothiazole_':    'c1nc2ccccc2s1',            
    'benzoxazole':      'c1nc2ccccc2o1',          
    'indazole':         'c1c([nH]nn2)c2ccc1',            
    'indazole_C':         'c1c([nX1]nn2)c2ccc1',      
}
    
def ring_substituents(mol, ring_atoms):
    substituents = set()
    mapnums = set()
    for idx in ring_atoms:
        atom = mol.GetAtomWithIdx(idx)
        for neighbor in atom.GetNeighbors():
            n_idx = neighbor.GetIdx()
            n_mapnums = neighbor.GetAtomMapNum()
            if n_idx not in ring_atoms and neighbor.GetAtomicNum() != 1:
                substituents.add(n_idx)
                mapnums.add(n_mapnums)
    return len(substituents), mapnums

def extract_functional_groups(mol):
    ids = []
    fg_list = []
    mapnum = []
    smarts_id = 0
    natoms = mol.GetNumAtoms()

    for name, smarts in fg_smarts_list.items():
        smarts_id += 1
        patt = Chem.MolFromSmarts(smarts)
        if patt is None:
            continue
            
        matches = mol.GetSubstructMatches(patt)

        for match in matches:
            atom_idxs = list(match)
            if len(atom_idxs) < 4:
                continue
            try:
                ring_atoms = atom_idxs  
                n_substituents, n_mapnums = ring_substituents(mol, ring_atoms)

                if 1 <= n_substituents <= 7:
                    ids.append(smarts_id)
                    fg_list.append(atom_idxs)
                    mapnum.append(n_mapnums)
            except Exception as e:
                #print(f'[ERROR] Check: {e}')
                continue

    return ids, fg_list, mapnum
    
for group_key, rows in tqdm(reaction_groups.items()):
    for r in rows:
        smiles = r['inputs'].split('.')[0]
        fg_site = r['class']
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"[Warning] Invalid SMILES: {smiles}")
            r['fgs'] = []
            continue
        ids, fgs, mapnum = extract_functional_groups(mol)
        filtered = distance.get_atoms_by_distance(mol, fg_site, min_dist=2)

        new_ids = []
        new_fgs = []
        new_mn = []
        
        for i in range(len(fgs)):
            if set(fgs[i]).issubset(filtered):
                new_ids.append(ids[i])
                new_fgs.append(fgs[i])
                new_mn.append(mapnum[i])
            
        r['fg_id'] = new_ids
        r['fg_mat'] = new_fgs
        r['fg_sub'] = new_mn

for group_key, rows in tqdm(reaction_groups_val.items()):
    for r in rows:
        smiles = r['inputs'].split('.')[0]
        fg_site = r['class']
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"[Warning] Invalid SMILES: {smiles}")
            r['fgs'] = []
            continue
        ids, fgs, mapnum = extract_functional_groups(mol)
        filtered = distance.get_atoms_by_distance(mol, fg_site, min_dist=2)

        new_ids = []
        new_fgs = []
        new_mn = []
        for i in range(len(fgs)):
            if set(fgs[i]).issubset(filtered):
                new_ids.append(ids[i])
                new_fgs.append(fgs[i])
                new_mn.append(mapnum[i])
            
        r['fg_id'] = new_ids
        r['fg_mat'] = new_fgs
        r['fg_sub'] = new_mn

100%|████████████████████████████████████████████████████████████████████████████████| 380/380 [00:05<00:00, 66.34it/s]


In [ ]:
#graph mixer
import augment_models.graph_mixer as graph_mixer
import csv

def chunk_dict(d, chunk_size):
    items = list(d.items())
    for i in range(0, len(items), chunk_size):
        yield dict(items[i:i+chunk_size])

def append_csv_chunk(data, filename, write_header=False):
    mode = 'w' if write_header else 'a'
    with open(filename, mode, newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'class', 'input>>output'])
        if write_header:
            writer.writeheader()
        
        formatted_data = []
        checker = 0
        for group_key, rows in data.items():
            for row in rows:
                new_row = {
                    'id': row['source_ids'],
                    'class': row['mixed_num'],
                    'input>>output': f"{row['inputs']}>>{row['output']}"
                }
                formatted_data.append(new_row)
                checker += 1

        writer.writerows(formatted_data)
    print(f"[Appended] {checker} rows → {filename}")

# ---------- Training ----------
train_csv_path = 'augmented_datasets/augmented_graph_mixer_train.csv'
print("[Start] Augmenting training data...")

for i, fg_chunk in enumerate(chunk_dict(reaction_groups, 25)):
    print(f"[Train] Chunk {i+1}")
    chunk_result = graph_mixer.graph_mixer_synt(fg_chunk, n_iter)
    if chunk_result:
        append_csv_chunk(chunk_result, train_csv_path, write_header=(i == 0))

# ---------- Validation ----------
val_csv_path = 'augmented_datasets/augmented_graph_mixer_val.csv'
print("[Start] Augmenting validation data...")

for i, fg_chunk in enumerate(chunk_dict(reaction_groups_val, 25)):
    print(f"[Val] Chunk {i+1}")
    chunk_result = graph_mixer.graph_mixer_synt(fg_chunk, n_iter)
    if chunk_result:
        append_csv_chunk(chunk_result, val_csv_path, write_header=(i == 0))

[Start] Augmenting training data...
[Train] Chunk 1


100%|███████████████████████████████████████████████████████████████████████████████| 25/25 [1:39:18<00:00, 238.32s/it]


[Appended] 3852 rows → augmented_datasets/augmented_graph_mixer_train.csv
[Train] Chunk 2


100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [11:39<00:00, 27.97s/it]


[Appended] 642 rows → augmented_datasets/augmented_graph_mixer_train.csv
[Train] Chunk 3


...

100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [00:55<00:00,  2.22s/it]


[Appended] 69 rows → augmented_datasets/augmented_graph_mixer_val.csv
[Val] Chunk 2


100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [04:47<00:00, 11.51s/it]


[Appended] 244 rows → augmented_datasets/augmented_graph_mixer_val.csv
[Val] Chunk 3


  8%|██████▋                                                                            | 2/25 [00:00<00:06,  3.52it/s]